# Week 1 – Data Acquisition, Cleaning, and Preprocessing

## Laptop Price Dataset

This notebook documents the complete data-cleaning workflow using the actual `laptopData (1).csv` dataset.

### Objectives
- Load and explore the raw dataset
- Identify missing values and duplicates
- Clean inconsistent and erroneous entries
- Detect outliers using the IQR method
- Extract useful numerical and categorical features
- Visualize the cleaned data
- Save an analysis-ready CSV


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


## 1. Load the Raw Dataset

The original dataset is loaded without changing it.

In [ ]:
df = pd.read_csv("../data/raw/laptopData (1).csv")

print("Dataset loaded successfully")
print("Shape:", df.shape)


In [ ]:
df.head()

## 2. Initial Data Exploration

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())


In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 3. Missing-Value Analysis

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})
missing_summary


In [ ]:
missing_rows = df.isna().any(axis=1).sum()
print("Rows containing at least one missing value:", missing_rows)


### Cleaning decision
Incomplete rows are removed because the missing records do not contain enough information to reliably reconstruct complete laptop specifications. This avoids inventing values for important fields such as price, CPU, GPU, or manufacturer.


## 4. Duplicate Analysis

In [ ]:
duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)


## 5. Basic Cleaning

In [ ]:
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

df = df.dropna().copy()
df = df.drop_duplicates().copy()

print("Shape after basic cleaning:", df.shape)


## 6. Convert Text-Based Numerical Fields

In [ ]:
df["Inches"] = pd.to_numeric(df["Inches"], errors="coerce")

df["Ram_GB"] = (
    df["Ram"].str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
)

df["Weight_kg"] = (
    df["Weight"].str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
)

df["CPU_GHz"] = (
    df["Cpu"].str.extract(r"(\d+(?:\.\d+)?)GHz")[0].astype(float)
)

df[["Inches", "Ram", "Ram_GB", "Weight", "Weight_kg", "CPU_GHz"]].head(10)


## 7. Detect and Correct an Erroneous Weight Entry

A laptop weight below 0.5 kg is treated as implausible in this dataset.

In [ ]:
invalid_weight = df["Weight_kg"] < 0.5

print("Potentially erroneous weight records:", invalid_weight.sum())
df.loc[invalid_weight, ["Company", "TypeName", "Weight", "Weight_kg", "Price"]]


In [ ]:
valid_median_weight = df.loc[~invalid_weight, "Weight_kg"].median()
df.loc[invalid_weight, "Weight_kg"] = valid_median_weight

print("Replacement median weight:", round(valid_median_weight, 2), "kg")


### Rationale
The value `0.0002 kg` is physically implausible for a laptop. The rest of the record is retained, while the invalid weight is replaced by the median of valid weights.


## 8. Extract Screen Features

In [ ]:
resolution = df["ScreenResolution"].str.extract(r"(\d{3,5})x(\d{3,5})")

df["ScreenWidth_px"] = pd.to_numeric(resolution[0], errors="coerce")
df["ScreenHeight_px"] = pd.to_numeric(resolution[1], errors="coerce")

df["Touchscreen"] = df["ScreenResolution"].str.contains(
    "Touchscreen", case=False, na=False
).astype(int)

df["IPS"] = df["ScreenResolution"].str.contains(
    "IPS", case=False, na=False
).astype(int)

df[["ScreenResolution", "ScreenWidth_px", "ScreenHeight_px",
    "Touchscreen", "IPS"]].head(10)


## 9. Extract CPU and GPU Brands

In [ ]:
df["CPU_Brand"] = df["Cpu"].str.split().str[0]
df["GPU_Brand"] = df["Gpu"].str.split().str[0]

df[["Cpu", "CPU_Brand", "Gpu", "GPU_Brand"]].head(10)


## 10. Outlier Detection Using IQR

The Interquartile Range method flags values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. A statistical outlier is not automatically an error; legitimate premium laptops can have high prices or high RAM.


In [ ]:
def iqr_analysis(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flags = ((series < lower) | (series > upper)).sum()
    return q1, q3, iqr, lower, upper, int(flags)

numeric_columns = ["Price", "Inches", "Ram_GB", "Weight_kg", "CPU_GHz"]

results = []
for column in numeric_columns:
    q1, q3, iqr, lower, upper, flags = iqr_analysis(df[column])
    results.append([column, q1, q3, iqr, lower, upper, flags])

iqr_table = pd.DataFrame(
    results,
    columns=["Variable", "Q1", "Q3", "IQR", "Lower Bound", "Upper Bound", "IQR Flags"]
)

iqr_table.round(2)


## 11. Visual Exploration

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["Price"], bins=30)
plt.title("Laptop Price Distribution")
plt.xlabel("Price")
plt.ylabel("Number of Laptops")
plt.show()


In [ ]:
plt.figure(figsize=(8,4))
plt.boxplot(df["Price"], vert=False)
plt.title("Laptop Price – IQR Outlier Inspection")
plt.xlabel("Price")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["Ram_GB"], df["Price"], alpha=0.45)
plt.title("RAM vs Laptop Price")
plt.xlabel("RAM (GB)")
plt.ylabel("Price")
plt.show()


In [ ]:
company_counts = df["Company"].value_counts().head(10).sort_values()

plt.figure(figsize=(8,5))
company_counts.plot(kind="barh")
plt.title("Top 10 Laptop Brands")
plt.xlabel("Number of Records")
plt.ylabel("Company")
plt.show()


## 12. Final Quality Check

In [ ]:
print("Final dataset shape:", df.shape)
print("\nRemaining missing values:")
print(df.isna().sum())
print("\nRemaining duplicate rows:", df.duplicated().sum())


In [ ]:
df[[
    "Price", "Inches", "Ram_GB", "Weight_kg", "CPU_GHz",
    "ScreenWidth_px", "ScreenHeight_px"
]].describe().T.round(2)


## 13. Save the Cleaned Dataset

In [ ]:
output_path = "../data/cleaned/laptop_cleaned_week1.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved to:", output_path)
print("Final shape:", df.shape)


## 14. Summary

The preprocessing workflow:
1. Loaded the original CSV.
2. Explored its structure and quality.
3. Identified and removed incomplete records.
4. Removed duplicate records.
5. Removed the unnecessary exported index.
6. Converted text-based numerical fields.
7. Corrected an implausible weight value.
8. Investigated outliers using IQR.
9. Extracted screen, CPU, and GPU features.
10. Visualized the cleaned data.
11. Saved the final dataset for future analysis.

### Final Reflection
Cleaning improves consistency and makes the dataset suitable for further analysis and machine learning. However, removing records and correcting values can affect downstream results, so each decision has been documented and made reproducible.
